In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer

from rag_fiqa.config import CHUNK_SIZE_TOKENS, EMBEDDING_MODEL_ID, EMBEDDING_MODEL_REVISION

In [ ]:
df = pd.read_json("../data/processed/fiqa/corpus.jsonl", lines=True)

In [ ]:
df

In [ ]:
df["text"].str.len().hist()

In [ ]:
df["text"].str.len().describe()

In [ ]:
# any paragraphs?
df["text"].str.contains("\n").sum()

In [ ]:
# sentences?
df["text"].str.contains("\\.").sum()

In [ ]:
df["title"].str.len().sum()

In [ ]:
# Token length distribution using the embedding model tokenizer (including [CLS] and [SEP])
tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_MODEL_ID,
    revision=EMBEDDING_MODEL_REVISION,
    local_files_only=True,
)
df["token_count"] = [
    len(ids) for ids in tokenizer(df["text"].tolist(), add_special_tokens=True, truncation=False)["input_ids"]
]
df["token_count"].describe()

In [ ]:
over_limit = (df["token_count"] > CHUNK_SIZE_TOKENS).sum()
total_docs = len(df)
print(
    f"Documents longer than {CHUNK_SIZE_TOKENS} tokens: {over_limit:,} / {total_docs:,} ({over_limit / total_docs:.1%})"
)

In [ ]:
ax = df["token_count"].clip(upper=1000).hist(bins=50, figsize=(8, 4))
ax.axvline(CHUNK_SIZE_TOKENS, color="red", linestyle="--", label=f"Max chunk tokens ({CHUNK_SIZE_TOKENS})")
ax.set_xlabel("Token count (clipped at 1,000)")
ax.set_ylabel("Number of documents")
ax.set_title("FiQA Document Token Length Distribution (all-MiniLM-L6-v2)")
ax.legend()
plt.tight_layout()